In [17]:
# This notebook constructs the parquet files result_48h.to_parquet and result_72h.to_parquet that contain the
# weather and water_level data in a format that is suitable for model training. Both files contain a timestamp column,
# that contains the date and hour when the water_level was reported.
# It also includes columns of this form: {weather_var or water_level}_{station_id}_lag_{hours}, where weather_var is one of the weather variables, 
# station_id is the id of the weather station, and hours is the number of hours of lag. the maximum lag in both files is 6h.
# Finally, the files contain a column water_level that contains the water level at the timestamp, and the column
# water_levels that contains the water levels for the 48h or 72h after the timestamp as well as the current water_level
# at the timestamp, depending on the file. 

import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/intermediate")
OUT_DIR = Path("../out")
PEGEL_PATH = DATA_DIR / "pegel_kirchentellinsfurt.csv"
WEATHER_HOURLY_PATH = DATA_DIR / "weather_data_hourly_1968_2026.csv"

def load_pegel(pegel_path=PEGEL_PATH, cutoff_year=1968):
    pegel = pd.read_csv(pegel_path, sep=";", usecols=["Messzeitpunkt", "Messwert"], dtype="string")
    pegel["timestamp"] = pd.to_datetime(pegel.pop("Messzeitpunkt"), format="%d.%m.%Y %H:%M", errors="coerce")
    pegel["water_level"] = pd.to_numeric(pegel.pop("Messwert").str.replace(",", ".", regex=False), errors="coerce")
    pegel = pegel.dropna(subset=["timestamp", "water_level"]).sort_values("timestamp")
    pegel["year"] = pegel["timestamp"].dt.year
    pegel["month"] = pegel["timestamp"].dt.month
    pegel = pegel[pegel["year"] >= cutoff_year]
    return pegel


def load_hourly_weather(weather_path=WEATHER_HOURLY_PATH):
    weather = pd.read_csv(weather_path, parse_dates=["timestamp"])
    weather["timestamp"] = weather["timestamp"].dt.floor("h")
    weather["station_id"] = weather["station_id"].astype(str).str.zfill(5)
    return weather

weather_data_hourly = load_hourly_weather()
pegel_data = load_pegel()

In [18]:
# Pivot hourly weather data wide so each station-variable is a feature column.
weather_data_hourly_pivoted = weather_data_hourly.pivot(index='timestamp', columns='station_id', values=['temperature_air_2m', 'precipitation_amount', 'humidity_relative', 'pressure_vapor']).reset_index()
weather_data_hourly_pivoted.columns = ['_'.join(map(str, col)) for col in weather_data_hourly_pivoted.columns]
weather_data_hourly_pivoted.rename(columns={'timestamp_': 'timestamp'}, inplace=True)

In [19]:
# See if hourly weather data is always reported exactly on the hour
repoting_times = weather_data_hourly['timestamp'].dt.minute.value_counts()
print(repoting_times)

timestamp
0    1226336
Name: count, dtype: int64


In [20]:
LAGS = 6

weather_data_hourly_pivoted = weather_data_hourly_pivoted.sort_values("timestamp")
pegel_data = pegel_data.sort_values("timestamp").reset_index(drop=True)

feature_frames = [pegel_data]

for h in range(1, LAGS + 1):
    target_time = pegel_data["timestamp"] - pd.Timedelta(hours=h)


    matched_weather = (pd.DataFrame({"target_time": target_time})
        .merge(weather_data_hourly_pivoted, left_on="target_time", right_on="timestamp", how="left")
        .drop(columns=["target_time", "timestamp"]))

    matched_weather.columns = [f"{col}_lag_{h}h" for col in matched_weather.columns]

    feature_frames.append(matched_weather)

    water_lag = pegel_data[["timestamp", "water_level"]].copy()
    water_lag["timestamp"] += pd.Timedelta(hours=h)
    water_lag = (pegel_data[["timestamp"]]
        .merge(water_lag, on="timestamp", how="left")
        .drop(columns="timestamp"))
    water_lag.columns = [f"water_level_lag_{h}h"]
    feature_frames.append(water_lag)

result = pd.concat(feature_frames, axis=1)
#result.to_parquet("../data/final/pegel_weather_6_hourly_lags.parquet", index=False)

assert len(result) == len(pegel_data)
assert "_row_id" not in result.columns
assert all(f"water_level_lag_{h}h" in result.columns for h in range(1, LAGS + 1))

In [21]:
# Check how many observations have missing values in the features
missing_values = result.isnull().sum()
print(missing_values)

timestamp                               0
water_level                             0
year                                    0
month                                   0
temperature_air_2m_02074_lag_1h    322570
                                    ...  
pressure_vapor_03278_lag_6h        316695
pressure_vapor_03402_lag_6h        309361
pressure_vapor_04931_lag_6h         84664
pressure_vapor_06275_lag_6h        322493
water_level_lag_6h                     10
Length: 130, dtype: int64


In [22]:
min_time = result["timestamp"].min()
max_time = result["timestamp"].max()
diff = max_time - min_time
print(f"Time range of the dataset: {min_time} to {max_time}, which is a difference of {diff}.")
print(f"Total hours in the time range: {diff.days * 24 + 20}, which is {len(result)} rows in the dataset.")

Time range of the dataset: 1968-01-01 00:00:00 to 2026-10-25 20:00:00, which is a difference of 21482 days 20:00:00.
Total hours in the time range: 515588, which is 515585 rows in the dataset.


In [23]:
# To predict hourly over the next 2-3 days, you could use vectors as outputs, that contain the water_levels for the next 48-72h.
# Construct a dataframe that has a column water_levels_48/water_levels_72 that contains a 47/72 long pd array of water_levels.

import numpy as np

def add_forward_water_levels(features, horizon_hours):
    """Return rows with complete hourly water-level sequences through the horizon."""
    timestamps = pd.DatetimeIndex(features["timestamp"])
    water_by_time = features.drop_duplicates("timestamp").set_index("timestamp")["water_level"].sort_index()
    windows = np.column_stack([
        water_by_time.reindex(timestamps + pd.Timedelta(hours=offset)).to_numpy()
        for offset in range(horizon_hours + 1)
    ])
    complete = ~np.isnan(windows).any(axis=1)
    labeled = features.loc[complete, :].copy()
    labeled["water_levels"] = windows[complete].tolist()
    return labeled.reset_index(drop=True)

result_48h = add_forward_water_levels(result, 48)
result_72h = add_forward_water_levels(result, 72)
result_48h.to_parquet("../data/final/pegel_weather_6_hourly_lags_forecast_48h.parquet", index=False)
result_72h.to_parquet("../data/final/pegel_weather_6_hourly_lags_forecast_72h.parquet", index=False)
del result

assert "water_level" in result_48h.columns and "water_level" in result_72h.columns
assert np.allclose(result_48h["water_levels"].map(lambda levels: levels[0]), result_48h["water_level"])
assert np.allclose(result_72h["water_levels"].map(lambda levels: levels[0]), result_72h["water_level"])
assert len(result_48h) >= len(result_72h)
assert all(len(levels) == 49 for levels in result_48h["water_levels"])
assert all(len(levels) == 73 for levels in result_72h["water_levels"])


In [24]:
print(result_48h.columns)

Index(['timestamp', 'water_level', 'year', 'month',
       'temperature_air_2m_02074_lag_1h', 'temperature_air_2m_03278_lag_1h',
       'temperature_air_2m_03402_lag_1h', 'temperature_air_2m_04931_lag_1h',
       'temperature_air_2m_06275_lag_1h', 'precipitation_amount_02074_lag_1h',
       ...
       'humidity_relative_03402_lag_6h', 'humidity_relative_04931_lag_6h',
       'humidity_relative_06275_lag_6h', 'pressure_vapor_02074_lag_6h',
       'pressure_vapor_03278_lag_6h', 'pressure_vapor_03402_lag_6h',
       'pressure_vapor_04931_lag_6h', 'pressure_vapor_06275_lag_6h',
       'water_level_lag_6h', 'water_levels'],
      dtype='str', length=131)
